# Building Block Preparation from ChEMBL

This notebook downloads small molecules from ChEMBL, applies physicochemical filters suitable for building blocks, standardizes structures, and removes duplicates.

## Workflow
1. Query ChEMBL with basic filters
2. Apply RDKit-based property filters
3. Export for standardization (ISIDA Standardizer)
4. Reload standardized structures and remove duplicates
5. Save final building block set

## 0. Install dependencies (run once)

In [1]:
# Run this cell once to install required packages
# Comment it out after first run
import subprocess
subprocess.run(['pip', 'install', 'chembl_webresource_client', 'pandas', 'tqdm'], check=True)

  Using cached chembl_webresource_client-0.10.9-py3-none-any.whl.metadata (1.4 kB)
  Using cached requests_cache-1.3.1-py3-none-any.whl.metadata (9.4 kB)
  Using cached easydict-1.13-py3-none-any.whl.metadata (4.2 kB)
  Using cached cattrs-26.1.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached url_normalize-2.2.1-py3-none-any.whl.metadata (5.6 kB)
Using cached chembl_webresource_client-0.10.9-py3-none-any.whl (55 kB)
Using cached requests_cache-1.3.1-py3-none-any.whl (70 kB)
Using cached cattrs-26.1.0-py3-none-any.whl (73 kB)
Using cached url_normalize-2.2.1-py3-none-any.whl (14 kB)
Using cached easydict-1.13-py3-none-any.whl (6.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [chembl_webresource_client]2m4/6 [requests-cache]


CompletedProcess(args=['pip', 'install', 'chembl_webresource_client', 'pandas', 'tqdm'], returncode=0)

## 1. Imports

In [2]:
import pandas as pd
from chembl_webresource_client.new_client import new_client
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from tqdm import tqdm
import time

print('All imports successful.')

All imports successful.


## 2. Query ChEMBL

We filter directly in the API query to reduce the number of records downloaded:
- `molecule_type = Small molecule`
- `structure_type = MOL` (single component, excludes salts and mixtures)
- `MW` between 100 and 300 Da (BB-appropriate size)
- Rotatable bonds <= 5

Note: logP, HBD, HBA are applied later in RDKit since API filtering on these can be unreliable.

In [3]:
molecule = new_client.molecule

print('Querying ChEMBL... this may take a few minutes.')

results = molecule.filter(
    molecule_type='Small molecule',
    structure_type='MOL',
    molecule_properties__mw_freebase__gte=100,   # MW >= 100 Da
    molecule_properties__mw_freebase__lte=300,   # MW <= 300 Da
    molecule_properties__rtb__lte=5,             # rotatable bonds <= 5
).only([
    'molecule_chembl_id',
    'molecule_structures',
    'molecule_properties'
])

print('Query sent. Fetching results...')

Querying ChEMBL... this may take a few minutes.
Query sent. Fetching results...


In [6]:
# Check count without downloading
results_check = molecule.filter(
    molecule_type='Small molecule',
    structure_type='MOL',
    molecule_properties__mw_freebase__gte=100,
    molecule_properties__mw_freebase__lte=300,
    molecule_properties__rtb__lte=5,
)

print(len(results_check))

300480


In [4]:
# Convert results to a list (this is where the actual download happens)
# This step can take several minutes depending on result size
results_list = list(results)
print(f'Downloaded {len(results_list)} compounds from ChEMBL.')

KeyboardInterrupt: 

## 3. Parse into DataFrame

In [ ]:
records = []

for mol in results_list:
    chembl_id = mol.get('molecule_chembl_id', None)
    
    # Get canonical SMILES
    structures = mol.get('molecule_structures', None)
    if structures is None:
        continue
    smiles = structures.get('canonical_smiles', None)
    if smiles is None:
        continue
    
    # Get properties
    props = mol.get('molecule_properties', None)
    if props is None:
        continue

    records.append({
        'chembl_id': chembl_id,
        'smiles': smiles,
        'mw': props.get('mw_freebase'),
        'alogp': props.get('alogp'),
        'hbd': props.get('hbd'),
        'hba': props.get('hba'),
        'rtb': props.get('rtb'),
    })

df = pd.DataFrame(records)
print(f'Parsed {len(df)} compounds.')
df.head()

## 4. RDKit-based filtering

We now apply additional filters using RDKit to ensure structural validity and drug-likeness:

| Filter | Value | Reason |
|---|---|---|
| Valid RDKit mol | required | removes unparseable SMILES |
| Organic only | C must be present | removes inorganic compounds |
| Allowed elements | C,H,N,O,S,P,F,Cl,Br,I | removes organometallics |
| logP | <= 5 | Ro5 |
| HBD | <= 5 | Ro5 |
| HBA | <= 10 | Ro5 |
| No isotopes | required | cleaner structures |

In [ ]:
ALLOWED_ATOMS = {6, 1, 7, 8, 16, 15, 9, 17, 35, 53}  
# C, H, N, O, S, P, F, Cl, Br, I (atomic numbers)

def passes_rdkit_filters(smiles):
    """
    Returns (True, mol) if the molecule passes all filters,
    (False, None) otherwise.
    """
    # Parse SMILES
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False, None
    
    # Must contain carbon (organic)
    atom_nums = {atom.GetAtomicNum() for atom in mol.GetAtoms()}
    if 6 not in atom_nums:
        return False, None
    
    # Only allowed elements
    if not atom_nums.issubset(ALLOWED_ATOMS):
        return False, None
    
    # No isotopes
    if any(atom.GetIsotope() != 0 for atom in mol.GetAtoms()):
        return False, None
    
    # Ro5 filters (logP, HBD, HBA)
    logp = Descriptors.MolLogP(mol)
    hbd  = rdMolDescriptors.CalcNumHBD(mol)
    hba  = rdMolDescriptors.CalcNumHBA(mol)
    
    if logp > 5:  return False, None
    if hbd  > 5:  return False, None
    if hba  > 10: return False, None

    return True, mol


# Apply filters
passed_smiles  = []
passed_chembl  = []
passed_mols    = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Filtering'):
    ok, mol = passes_rdkit_filters(row['smiles'])
    if ok:
        passed_smiles.append(row['smiles'])
        passed_chembl.append(row['chembl_id'])
        passed_mols.append(mol)

df_filtered = pd.DataFrame({
    'chembl_id': passed_chembl,
    'smiles': passed_smiles
})

print(f'Before filtering: {len(df)} compounds')
print(f'After filtering:  {len(df_filtered)} compounds')

## 5. Export for ISIDA Standardizer

Export SMILES to a text file. You will upload this to:
https://chematlas.chimie.unistra.fr/WebTools/standardizer.php

After standardization, download the result and continue from Section 6.

In [ ]:
# Export SMILES list for standardizer
output_path = 'chembl_filtered_for_standardization.smi'

with open(output_path, 'w') as f:
    for idx, row in df_filtered.iterrows():
        f.write(f"{row['smiles']} {row['chembl_id']}\n")

print(f'Exported {len(df_filtered)} SMILES to {output_path}')
print('Upload this file to the ISIDA Standardizer, download the result, then continue below.')

## 6. Reload standardized structures and remove duplicates

After downloading the standardized file from the ISIDA tool, set the correct filename below and run this section.

Duplicate detection is based on canonical SMILES — two molecules are considered identical if their canonical SMILES match after standardization.

In [ ]:
# Set this to the filename you downloaded from the standardizer
standardized_file = 'chembl_standardized.smi'  # <-- update this

# Read standardized SMILES
df_std = pd.read_csv(
    standardized_file,
    sep='\s+',
    header=None,
    names=['smiles', 'chembl_id']
)

print(f'Loaded {len(df_std)} standardized compounds.')
df_std.head()

In [ ]:
# Generate canonical SMILES using RDKit for duplicate detection
canonical = []
valid_idx = []

for idx, row in tqdm(df_std.iterrows(), total=len(df_std), desc='Canonicalizing'):
    mol = Chem.MolFromSmiles(row['smiles'])
    if mol is not None:
        canonical.append(Chem.MolToSmiles(mol))
        valid_idx.append(idx)

df_std = df_std.loc[valid_idx].copy()
df_std['canonical_smiles'] = canonical

# Remove duplicates based on canonical SMILES
before_dedup = len(df_std)
df_dedup = df_std.drop_duplicates(subset='canonical_smiles', keep='first').reset_index(drop=True)
after_dedup = len(df_dedup)

print(f'Before deduplication: {before_dedup} compounds')
print(f'After deduplication:  {after_dedup} compounds')
print(f'Duplicates removed:   {before_dedup - after_dedup}')

## 7. Save final building block set

In [ ]:
final_output = 'building_blocks_final.smi'

df_dedup[['canonical_smiles', 'chembl_id']].to_csv(
    final_output,
    sep=' ',
    header=False,
    index=False
)

print(f'Final building block set saved: {len(df_dedup)} compounds -> {final_output}')

## 8. Summary statistics

Quick overview of the final building block set.

In [ ]:
from rdkit.Chem import Descriptors

mws, logps, hbds, hbas = [], [], [], []

for smi in tqdm(df_dedup['canonical_smiles'], desc='Computing properties'):
    mol = Chem.MolFromSmiles(smi)
    if mol:
        mws.append(Descriptors.MolWt(mol))
        logps.append(Descriptors.MolLogP(mol))
        hbds.append(rdMolDescriptors.CalcNumHBD(mol))
        hbas.append(rdMolDescriptors.CalcNumHBA(mol))

summary = pd.DataFrame({'MW': mws, 'logP': logps, 'HBD': hbds, 'HBA': hbas})
print('\nFinal building block set - property summary:')
summary.describe().round(2)